# DiabPred – Full Walkthrough Notebook

This notebook walks through the entire DiabPred pipeline step by step:
1. Data loading and exploratory analysis
2. Preprocessing pipeline
3. Training all classifiers
4. Evaluating and comparing models
5. Predicting for a new patient
6. Generating publication figures

> **Run all cells top-to-bottom.** Every cell is self-contained and explained.

## 0. Setup – install and import

In [ ]:
# If running in Google Colab, uncomment:
# !git clone https://github.com/yourusername/diabpred.git
# %cd diabpred
# !pip install -e . -q

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print('All imports OK')

## 1. Load the dataset

In [ ]:
from diabpred.data import load_dataset, describe_dataset, get_feature_names

df = load_dataset()
print(f'Shape: {df.shape}  |  Diabetic: {df.Outcome.mean():.1%}')
df.head()

In [ ]:
# Summary statistics including zero counts (zeros = missing values in this dataset)
describe_dataset(df)

In [ ]:
# Class distribution
counts = df.Outcome.value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
axes[0].bar(['Non-diabetic', 'Diabetic'], counts.values, color=['#378ADD', '#D85A30'], width=0.5)
axes[0].set_title('Class distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=11)

# Feature distributions by class
feat = 'Glucose'
df[df.Outcome == 0][feat].plot(kind='hist', bins=25, alpha=0.6, color='#378ADD',
                                label='Non-diabetic', ax=axes[1])
df[df.Outcome == 1][feat].plot(kind='hist', bins=25, alpha=0.6, color='#D85A30',
                                label='Diabetic', ax=axes[1])
axes[1].set_title(f'{feat} distribution by class')
axes[1].set_xlabel(feat)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
import matplotlib.colors as mcolors

corr = df.corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()
print('Glucose has the highest correlation with Outcome:', round(corr['Outcome']['Glucose'], 3))

## 2. Preprocess

In [ ]:
from diabpred.data import preprocess

X_train, X_test, y_train, y_test, scaler = preprocess(df, random_state=42)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Train class balance: {y_train.mean():.1%} positive')
print(f'Test class balance:  {y_test.mean():.1%} positive')
print(f'Scaler mean (Glucose): {scaler.mean_[1]:.2f}')

## 3. Train all classifiers

In [ ]:
from diabpred.models import train_all_models, list_available_models
import time

print(f'Models to train: {list_available_models()}\n')
t0 = time.time()
trained_models = train_all_models(X_train, y_train, verbose=True)
print(f'\nTotal training time: {time.time() - t0:.2f}s')

## 4. Evaluate and compare

In [ ]:
from diabpred.evaluate import evaluate_model, full_benchmark, print_benchmark

# Quick look at one model
metrics = evaluate_model(trained_models['Random Forest'], X_test, y_test)
print('Random Forest metrics:')
for k, v in metrics.items():
    print(f'  {k:<25} {v}')

In [ ]:
# Full benchmark table (this is Table 1 in the paper)
bench_df = full_benchmark(trained_models, X_train, X_test, y_train, y_test, cv=5)
print_benchmark(bench_df)

In [ ]:
# Show just the key columns cleanly
display_cols = ['accuracy', 'f1', 'roc_auc', 'mcc', 'cv_roc_auc_mean', 'cv_roc_auc_std']
bench_df[display_cols].style.highlight_max(color='#c6efce', axis=0).format('{:.4f}')

## 5. Visualizations

In [ ]:
# ROC curves — inline in the notebook
from sklearn.metrics import roc_curve, auc

PALETTE = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
fig, ax = plt.subplots(figsize=(8, 6))
for (name, model), color in zip(trained_models.items(), PALETTE):
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random (0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves – All Models'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance for best model
from sklearn.inspection import permutation_importance

best_name = bench_df['roc_auc'].idxmax()
best_model = trained_models[best_name]
print(f'Best model: {best_name}')

result = permutation_importance(best_model, X_test, y_test,
                                n_repeats=10, random_state=42, scoring='roc_auc')
names = get_feature_names()
idx = result.importances_mean.argsort()[::-1]

fig, ax = plt.subplots(figsize=(8,5))
ax.barh([names[i] for i in idx], result.importances_mean[idx],
        xerr=result.importances_std[idx], color='#378ADD', alpha=0.85, capsize=3)
ax.invert_yaxis()
ax.set_xlabel('Mean decrease in ROC-AUC')
ax.set_title(f'Permutation Feature Importance – {best_name}')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Predict for a new patient

In [ ]:
from diabpred.predict import predict, predict_proba

# High-risk patient (matches first row of the original Pima dataset)
patient = {
    'Pregnancies': 6,
    'Glucose': 148,
    'BloodPressure': 72,
    'SkinThickness': 35,
    'Insulin': 0,
    'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age': 50
}

result = predict(patient, best_model, scaler)
print('=== Single Model Prediction ===')
for k, v in result.items():
    print(f'  {k:<20} {v}')

In [ ]:
# Ensemble view from all models
probs = predict_proba(patient, trained_models, scaler)
print('=== Ensemble Probabilities ===')
for model_name, prob in probs.items():
    bar = '█' * int(prob * 30)
    marker = ' ← ensemble mean' if model_name == 'ensemble_mean' else ''
    print(f'  {model_name:<28} {prob:.1%}  {bar}{marker}')

## 7. Save everything

In [ ]:
from diabpred.models import save_model
from pathlib import Path

Path('outputs/models').mkdir(parents=True, exist_ok=True)
for name, model in trained_models.items():
    path = save_model(model, name, 'outputs/models')
    print(f'Saved: {path}')

bench_df.to_csv('outputs/benchmark.csv')
print('Benchmark saved: outputs/benchmark.csv')